***SKIN CANCER***

**OBJECTIVES**
To check all models and identify which have better accuracy for software usage.

In [1]:
!pip install -q kaggle


In [2]:
pip install kagglehub torch torchvision scikit-learn xgboost thop pandas numpy pillow


In [3]:
!pip install thop

In [4]:
import os
import time
import glob
import pandas as pd
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models
import kagglehub

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from xgboost import XGBClassifier
from thop import profile

# -------------------------------------------------------------
# 1. Dataset Downloading & Custom Recursive Loader
# -------------------------------------------------------------
print("Downloading / Accessing dataset...")
root_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")
print("Path to dataset files:", root_path)

# Collect all image paths recursively across Train/Test subdirectories
valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp')
all_file_paths = []

for root, _, files in os.walk(root_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_file_paths.append(os.path.join(root, file))

# Extract class names from directory structure
# Assumes structure like: .../Train/melanoma/img1.jpg or .../melanoma/img1.jpg
class_set = set()
for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    class_set.add(parent_dir)

# Remove generic top-level folder names if captured
class_set.discard('Train')
class_set.discard('Test')

class_names = sorted(list(class_set))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
num_classes = len(class_names)

print(f"Detected {num_classes} classes: {class_names}")

# Filter valid images and assign numerical labels
file_paths = []
labels = []

for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    if parent_dir in class_to_idx:
        file_paths.append(p)
        labels.append(class_to_idx[parent_dir])

print(f"Total valid images loaded: {len(file_paths)}")

# Train / Validation Split (80/20 Stratified)
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# PyTorch Custom Dataset
class SkinCancerDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            img = self.transform(img)
        return img, label

transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_val = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_dataset = SkinCancerDataset(train_paths, train_labels, transform=transform_train)
val_dataset = SkinCancerDataset(val_paths, val_labels, transform=transform_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# -------------------------------------------------------------
# 2. Model Factory & Evaluation Routines
# -------------------------------------------------------------
def get_model(model_name, num_classes):
    weights = models.AlexNet_Weights.DEFAULT if model_name == 'AlexNet' else 'DEFAULT'
    if model_name == 'AlexNet':
        model = models.alexnet(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'VGG16':
        model = models.vgg16(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'VGG19':
        model = models.vgg19(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == 'ResNet18':
        model = models.resnet18(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'ResNet50':
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'ResNet101':
        model = models.resnet101(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == 'DenseNet121':
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
    elif model_name == 'EfficientNet-B0':
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    return model

def compute_metrics(y_true, y_pred, y_prob):
    acc = accuracy_score(y_true, y_pred) * 100
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    rec = recall_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0) * 100
    try:
        auc = roc_auc_score(y_true, y_prob, multi_class='ovr', average='weighted') * 100
    except Exception:
        auc = 0.0
    return round(acc, 2), round(prec, 2), round(rec, 2), round(f1, 2), round(auc, 2)

# -------------------------------------------------------------
# 3. Table 1 & Table 3 Pipeline
# -------------------------------------------------------------
models_t1 = ['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'ResNet101', 'DenseNet121', 'EfficientNet-B0']
models_t3 = ['AlexNet', 'VGG16', 'VGG19', 'ResNet18', 'ResNet50', 'DenseNet121', 'EfficientNet-B0']

res_t1 = []
res_t3 = []
epochs = 2  # Adjust epoch count as needed

for m_name in models_t1:
    print(f"\nTraining: {m_name}")
    model = get_model(m_name, num_classes).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    for epoch in range(epochs):
        model.train()
        for imgs, lbls in train_loader:
            imgs, lbls = imgs.to(device), lbls.to(device)
            optimizer.zero_grad()
            outputs = model(imgs)
            loss = criterion(outputs, lbls)
            loss.backward()
            optimizer.step()

    model.eval()
    all_preds, all_probs, all_targets = [], [], []
    start_time = time.time()
    with torch.no_grad():
        for imgs, lbls in val_loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())
            all_targets.extend(lbls.numpy())

    total_time = time.time() - start_time
    avg_inf_ms = (total_time / len(val_dataset)) * 1000

    acc, prec, rec, f1, auc = compute_metrics(all_targets, all_preds, np.array(all_probs))
    res_t1.append({
        'Model': m_name, 'Accuracy (%)': acc, 'Precision (%)': prec,
        'Recall (%)': rec, 'F1-Score (%)': f1, 'AUC (%)': auc
    })

    if m_name in models_t3:
        dummy_input = torch.randn(1, 3, 224, 224).to(device)
        flops, params = profile(model, inputs=(dummy_input,), verbose=False)
        torch.save(model.state_dict(), 'temp.pth')
        size_mb = round(os.path.getsize('temp.pth') / (1024 * 1024), 2)
        if os.path.exists('temp.pth'):
            os.remove('temp.pth')

        res_t3.append({
            'Model': m_name, 'Parameters (M)': round(params / 1e6, 2),
            'Model Size (MB)': size_mb, 'FLOPs (G)': round(flops / 1e9, 2),
            'Inference Time (ms)': round(avg_inf_ms, 2), 'Accuracy (%)': acc
        })

# -------------------------------------------------------------
# 4. Table 2 Pipeline (Deep Feature Extraction)
# -------------------------------------------------------------
print("\nExtracting Deep Features with ResNet50...")
extractor = models.resnet50(weights='DEFAULT')
extractor.fc = nn.Identity()
extractor = extractor.to(device)
extractor.eval()

def get_deep_features(loader):
    f_list, l_list = [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            feats = extractor(imgs)
            f_list.append(feats.cpu().numpy())
            l_list.append(lbls.numpy())
    return np.vstack(f_list), np.concatenate(l_list)

X_tr, y_tr = get_deep_features(train_loader)
X_va, y_va = get_deep_features(val_loader)

classifiers = {
    'Logistic Regression': LogisticRegression(max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(),
    'Random Forest': RandomForestClassifier(n_estimators=100),
    'K-Nearest Neighbors (KNN)': KNeighborsClassifier(n_neighbors=5),
    'Linear SVM': SVC(kernel='linear', probability=True),
    'RBF-SVM': SVC(kernel='rbf', probability=True),
    'XGBoost': XGBClassifier(eval_metric='mlogloss')
}

res_t2 = []
for c_name, clf in classifiers.items():
    clf.fit(X_tr, y_tr)
    p_preds = clf.predict(X_va)
    p_probs = clf.predict_proba(X_va)
    acc, prec, rec, f1, auc = compute_metrics(y_va, p_preds, p_probs)
    res_t2.append({
        'Feature Extractor': 'Deep Features', 'Classifier': c_name,
        'Accuracy (%)': acc, 'Precision (%)': prec,
        'Recall (%)': rec, 'F1-Score (%)': f1, 'AUC (%)': auc
    })

# -------------------------------------------------------------
# 5. Printable Results
# -------------------------------------------------------------
print("\n### Table 1. Comparison of Transfer Learning Models")
print(pd.DataFrame(res_t1).to_markdown(index=False))

print("\n### Table 2. Comparison of Different Classifiers")
print(pd.DataFrame(res_t2).to_markdown(index=False))

print("\n### Table 3. Computational Efficiency Comparison")
print(pd.DataFrame(res_t3).to_markdown(index=False))

Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Path to dataset files: /kaggle/input/skin-cancer9-classesisic
Detected 9 classes: ['actinic keratosis', 'basal cell carcinoma', 'dermatofibroma', 'melanoma', 'nevus', 'pigmented benign keratosis', 'seborrheic keratosis', 'squamous cell carcinoma', 'vascular lesion']
Total valid images loaded: 2357
Using device: cuda

Training: AlexNet
Downloading: "https://download.pytorch.org/models/alexnet-owt-7be5be79.pth" to /root/.cache/torch/hub/checkpoints/alexnet-owt-7be5be79.pth


100%|██████████| 233M/233M [00:01<00:00, 177MB/s]



Training: VGG16
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:06<00:00, 88.6MB/s]



Training: VGG19
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


...


Training: DenseNet121
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/checkpoints/densenet121-a639ec97.pth


100%|██████████| 30.8M/30.8M [00:00<00:00, 94.2MB/s]



Training: EfficientNet-B0
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 161MB/s]



Extracting Deep Features with ResNet50...

### Table 1. Comparison of Transfer Learning Models
| Model           |   Accuracy (%) |   Precision (%) |   Recall (%) |   F1-Score (%) |   AUC (%) |
|:----------------|---------------:|----------------:|-------------:|---------------:|----------:|
| AlexNet         |          64.19 |           65.02 |        64.19 |          63.03 |     92.31 |
| VGG16           |          55.72 |           63.82 |        55.72 |          53.9  |     89.31 |
| VGG19           |          63.35 |           62.46 |        63.35 |          62.14 |     90.9  |
| ResNet18        |          68.01 |           67.41 |        68.01 |          65.43 |     93.66 |
| ResNet50        |          68.86 |           67.04 |        68.86 |          66.27 |     93.08 |
| ResNet101       |          69.92 |           68.28 |        69.92 |          68.18 |     93.51 |
| DenseNet121     |          67.37 |           63.73 |        67.37 |          64.94 |     94.45 |
| EfficientNe

In [5]:
import os
import cv2
import time
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, models

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, balanced_accuracy_score
)
import kagglehub

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------------------------------------------
# 1. Dataset Downloading & Path Scanning
# -------------------------------------------------------------
print("Accessing dataset...")
root_path = kagglehub.dataset_download("nodoubttome/skin-cancer9-classesisic")

valid_extensions = ('.jpg', '.jpeg', '.png')
all_file_paths = []
for root, _, files in os.walk(root_path):
    for file in files:
        if file.lower().endswith(valid_extensions):
            all_file_paths.append(os.path.join(root, file))

class_set = {os.path.basename(os.path.dirname(p)) for p in all_file_paths}
class_set.discard('Train')
class_set.discard('Test')
class_names = sorted(list(class_set))
class_to_idx = {cls_name: idx for idx, cls_name in enumerate(class_names)}
num_classes = len(class_names)

file_paths, labels = [], []
for p in all_file_paths:
    parent_dir = os.path.basename(os.path.dirname(p))
    if parent_dir in class_to_idx:
        file_paths.append(p)
        labels.append(class_to_idx[parent_dir])

# Stratified 80/20 train/validation split
train_paths, val_paths, train_labels, val_labels = train_test_split(
    file_paths, labels, test_size=0.2, random_state=42, stratify=labels
)

# -------------------------------------------------------------
# 2. Spatial Image Filtering Implementations
# -------------------------------------------------------------
def apply_image_filter(img_np, filter_type):
    if filter_type == 'No Filter':
        return img_np
    elif filter_type == 'Average':
        return cv2.blur(img_np, (5, 5))
    elif filter_type == 'Gaussian':
        return cv2.GaussianBlur(img_np, (5, 5), 0)
    elif filter_type == 'Median':
        return cv2.medianBlur(img_np, 5)
    elif filter_type == 'Sharpening':
        kernel = np.array([[0, -1, 0], [-1, 5, -1], [0, -1, 0]])
        return cv2.filter2D(img_np, -1, kernel)
    elif filter_type == 'Sobel':
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        sobelx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
        sobely = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
        sobel_mag = cv2.magnitude(sobelx, sobely)
        sobel_norm = cv2.normalize(sobel_mag, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        return cv2.cvtColor(sobel_norm, cv2.COLOR_GRAY2RGB)
    return img_np

class FilteredSkinDataset(Dataset):
    def __init__(self, paths, labels, filter_type='No Filter', transform=None):
        self.paths = paths
        self.labels = labels
        self.filter_type = filter_type
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img_np = np.array(Image.open(self.paths[idx]).convert('RGB'))
        filtered_np = apply_image_filter(img_np, self.filter_type)
        img_pil = Image.fromarray(filtered_np)
        if self.transform:
            img_pil = self.transform(img_pil)
        return img_pil, self.labels[idx]

base_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# -------------------------------------------------------------
# 3. Model Architecture Loader & Metrics Computation
# -------------------------------------------------------------
def get_selected_model(model_name, num_classes):
    if model_name == 'ResNet101':
        m = models.resnet101(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == 'ResNet50':
        m = models.resnet50(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    elif model_name == 'ResNet18':
        m = models.resnet18(weights='DEFAULT')
        m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

def evaluate_model(model, loader):
    model.eval()
    preds_all, probs_all, targets_all = [], [], []
    with torch.no_grad():
        for imgs, lbls in loader:
            imgs = imgs.to(device)
            outputs = model(imgs)
            probs = torch.softmax(outputs, dim=1)
            preds = torch.argmax(probs, dim=1)
            preds_all.extend(preds.cpu().numpy())
            probs_all.extend(probs.cpu().numpy())
            targets_all.extend(lbls.numpy())

    preds_all, probs_all, targets_all = np.array(preds_all), np.array(probs_all), np.array(targets_all)

    acc = accuracy_score(targets_all, preds_all) * 100
    prec = precision_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    rec = recall_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    f1 = f1_score(targets_all, preds_all, average='weighted', zero_division=0) * 100
    macro_f1 = f1_score(targets_all, preds_all, average='macro', zero_division=0) * 100

    try:
        auc = roc_auc_score(targets_all, probs_all, multi_class='ovr', average='weighted') * 100
    except Exception:
        auc = 0.0

    return {
        'Accuracy': round(acc, 2),
        'Precision': round(prec, 2),
        'Recall': round(rec, 2),
        'F1-score': round(f1, 2),
        'Macro-F1': round(macro_f1, 2),
        'AUC': round(auc, 2)
    }

# -------------------------------------------------------------
# 4. Running Benchmarks
# -------------------------------------------------------------
selected_models = ['ResNet101', 'ResNet50', 'ResNet18']
filters = ['No Filter', 'Average', 'Gaussian', 'Median', 'Sharpening', 'Sobel']
experiment_results = []
epochs = 2

for m_name in selected_models:
    for f_type in filters:
        print(f"Evaluating Model: {m_name} | Filter: {f_type}")

        tr_ds = FilteredSkinDataset(train_paths, train_labels, filter_type=f_type, transform=base_transform)
        va_ds = FilteredSkinDataset(val_paths, val_labels, filter_type=f_type, transform=base_transform)

        tr_loader = DataLoader(tr_ds, batch_size=32, shuffle=True, num_workers=2)
        va_loader = DataLoader(va_ds, batch_size=32, shuffle=False, num_workers=2)

        model = get_selected_model(m_name, num_classes).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(model.parameters(), lr=1e-4)

        for epoch in range(epochs):
            model.train()
            for imgs, lbls in tr_loader:
                imgs, lbls = imgs.to(device), lbls.to(device)
                optimizer.zero_grad()
                outputs = model(imgs)
                loss = criterion(outputs, lbls)
                loss.backward()
                optimizer.step()

        metrics = evaluate_model(model, va_loader)
        metrics['Model'] = m_name
        metrics['Filter'] = f_type
        experiment_results.append(metrics)

df_results = pd.DataFrame(experiment_results)[['Model', 'Filter', 'Accuracy', 'Precision', 'Recall', 'F1-score', 'Macro-F1', 'AUC']]
print("\n" + "="*60)
print(df_results.to_markdown(index=False))

Accessing dataset...
Using Colab cache for faster access to the 'skin-cancer9-classesisic' dataset.
Evaluating Model: ResNet101 | Filter: No Filter
Evaluating Model: ResNet101 | Filter: Average
Evaluating Model: ResNet101 | Filter: Gaussian
Evaluating Model: ResNet101 | Filter: Median
Evaluating Model: ResNet101 | Filter: Sharpening
Evaluating Model: ResNet101 | Filter: Sobel
Evaluating Model: ResNet50 | Filter: No Filter
Evaluating Model: ResNet50 | Filter: Average
Evaluating Model: ResNet50 | Filter: Gaussian
Evaluating Model: ResNet50 | Filter: Median
Evaluating Model: ResNet50 | Filter: Sharpening
Evaluating Model: ResNet50 | Filter: Sobel
Evaluating Model: ResNet18 | Filter: No Filter
Evaluating Model: ResNet18 | Filter: Average
Evaluating Model: ResNet18 | Filter: Gaussian
Evaluating Model: ResNet18 | Filter: Median
Evaluating Model: ResNet18 | Filter: Sharpening
Evaluating Model: ResNet18 | Filter: Sobel

| Model     | Filter     |   Accuracy |   Precision |   Recall |   F1-scor